# Acceptance: a finite-difference Hessian for the Laplace metric

**This is a sign-off page, not a tutorial.** It exercises `plans/laplace-fd-hessian.plan.md`.
`wf.laplace_metric` used to take only the exact Hessian, `jax.hessian`, which is forward-mode over
reverse-mode autodiff. summer4's `default_solver()` solves with diffrax's default
`RecursiveCheckpointAdjoint`, a `custom_vjp` that forward mode cannot enter, so the metric failed
on any posterior that used it. `laplace_metric(hessian="auto")`, the new default, falls back to
central differences of the reverse-mode gradient. The automated checks live in
`tests/test_calib_laplace_fd.py` and `tests/test_calib_workflow_tune.py`.

The model is a three-compartment SIR fitted to nine prevalence points, with log-normal priors on
the infection and recovery rates, solved with the default adaptive solver. A second posterior,
identical except for `adjoint=diffrax.DirectAdjoint()`, gives the exact Hessian to compare with.

In [ ]:
import time

import diffrax
import jax
import jax.numpy as jnp
import numpy as np
import numpyro
import numpyro.distributions as dist
import pandas as pd
from numpyro.infer import MCMC, NUTS

from summer4 import (
    Compartments,
    FlowModel,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.epi.calibration import NumpyroPosterior
from summer4.epi.calibration import workflow as wf
from summer4.solvers import Diffrax, default_solver

state = Property("state", ("S", "I", "R"))
fm = FlowModel(PropertyMap.from_property(state))
fm.add_flow(TransitionFlow("infection", state["S"], state["I"], Param("infection")))
fm.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("recovery")))
cm = fm.compile()
y0 = PropertyData.wrap(cm.pmap, np.array([999.0, 1.0, 0.0]))
plan = SavePlan(requests={"I": SaveRequest(Compartments(where=state["I"]), ts=np.linspace(0, 40, 9))})


def solve(beta, gamma, solver):
    result = cm.run(
        {"infection": beta, "recovery": gamma}, y0, t0=0.0, t1=40.0, dt=0.5, save=plan, solver=solver
    )
    raw = result["I"].values
    return jnp.ravel(getattr(raw, "data", raw))


obs = np.asarray(solve(0.35, 0.1, default_solver()))


def make_post(solver):
    def model():
        beta = numpyro.sample("beta", dist.LogNormal(np.log(0.3), 0.5))
        gamma = numpyro.sample("gamma", dist.LogNormal(np.log(0.1), 0.5))
        numpyro.sample("y", dist.Normal(solve(beta, gamma, solver), 10.0), obs=obs)

    return NumpyroPosterior(model)


base = default_solver()
direct = Diffrax(base.solver, stepsize_controller=base.stepsize_controller, adjoint=diffrax.DirectAdjoint())
post = make_post(base)
post_direct = make_post(direct)
z = {"beta": np.log(0.35), "gamma": np.log(0.1)}  # unconstrained (log) true values
print(base)

## A. The exact Hessian on the default solver

`hessian="exact"` still uses `jax.hessian`. On this posterior it now raises a `ValueError` that
says why and names both fixes, chained from JAX's `TypeError`.

In [ ]:
try:
    wf.laplace_metric(post, z, hessian="exact")
except ValueError as exc:
    print("ValueError:", exc)
    print("caused by:", type(exc.__cause__).__name__, str(exc.__cause__).splitlines()[0])

## B. `"auto"` falls back to finite differences

The default `hessian="auto"` tries the exact Hessian and, on that failure only, takes central
differences of the reverse-mode gradient: 2n gradient evaluations (here 4) in one `jax.vmap`,
coordinate `i` stepped by `fd_step * max(1, |x_i|)`. `metric.hessian` records the method used.

In [ ]:
metric = wf.laplace_metric(post, z)
exact = wf.laplace_metric(post_direct, z, hessian="exact")
print("auto used:", metric.hessian, "| DirectAdjoint used:", exact.hessian)
names = list(metric.element_names)
pd.concat(
    {
        "fd (default solver)": pd.DataFrame(metric.covariance, index=names, columns=names),
        "exact (DirectAdjoint)": pd.DataFrame(exact.covariance, index=names, columns=names),
    },
    axis=1,
)

## C. Agreement with the exact Hessian across step sizes

The largest relative difference between the fd and exact covariances, for steps from `1e-2` to
`1e-7`. Too large a step adds truncation error (it falls as the step squared); too small a step
amplifies the gradient's rounding and solver noise by `1 / fd_step`. The default `1e-4` sits in
the flat middle.

In [ ]:
rows = []
for step in (1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7):
    m = wf.laplace_metric(post, z, hessian="fd", fd_step=step)
    rows.append(
        {
            "fd_step": step,
            "max rel diff (covariance)": float(np.max(np.abs(m.covariance - exact.covariance) / np.abs(exact.covariance))),
            "max rel diff (eigenvalues)": float(np.max(np.abs(m.eigenvalues - exact.eigenvalues) / np.abs(exact.eigenvalues))),
        }
    )
pd.DataFrame(rows).set_index("fd_step").map("{:.1e}".format)

## D. Cost

`laplace_metric` wall time (each call traces and compiles its own function, so this is what a
caller pays once per metric), and the median time of one jitted log density plus gradient for
each solver (`wf.time_gradient`). The fd metric costs a few extra gradients once; the
`DirectAdjoint` workaround makes every gradient, and so every NUTS step, slower.

In [ ]:
def wall(fn, repeats=3):
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        fn()
        times.append(time.perf_counter() - start)
    return float(np.median(times))


pd.Series(
    {
        "laplace_metric, auto -> fd, default solver (s)": wall(lambda: wf.laplace_metric(post, z)),
        "laplace_metric, exact, DirectAdjoint (s)": wall(lambda: wf.laplace_metric(post_direct, z, hessian="exact")),
        "one gradient, default solver (ms)": 1e3 * wf.time_gradient(post, z, repeats=50),
        "one gradient, DirectAdjoint (ms)": 1e3 * wf.time_gradient(post_direct, z, repeats=50),
    },
    name="median",
).to_frame().round(3)

## E. A short NUTS run from the fd metric

The fd metric starts a dense NUTS kernel on the default-solver posterior, never adapted
(`adapt_mass_matrix=False`), so warm-up tunes only the step size. The posterior standard
deviations in unconstrained space should be close to the Laplace ones for this nearly Gaussian
posterior.

In [ ]:
setting = wf.NutsSetting(
    "laplace fixed", {"dense_mass": True, "adapt_mass_matrix": False}, laplace=True
)
init = wf.Candidates.from_z(post, {k: np.atleast_1d(v) for k, v in z.items()})
mcmc = setting.make_mcmc(post.numpyro_model(), num_samples=300, num_chains=2, metric=metric)(150)
mcmc.run(jax.random.PRNGKey(0), init_params=init.init_params(2, jitter=0.05, seed=0))
draws = mcmc.get_samples(group_by_chain=False)
divergences = int(np.sum(mcmc.get_extra_fields()["diverging"]))
posterior_std = np.array([np.std(np.log(np.asarray(draws[s]))) for s in metric.sites])
print("divergences:", divergences)
pd.DataFrame({"Laplace std (fd)": metric.std, "posterior std": posterior_std}, index=names)

**Sign-off:** section A shows the exact Hessian's failure and the new message; B shows `"auto"`
choosing fd; C shows the fd covariance agreeing with the `DirectAdjoint` exact one to about
`2e-7` relative over several decades of step size; D shows the cost; E shows NUTS running from the fd metric with
scales that match the posterior's.